<a href="https://colab.research.google.com/github/Jalilnkh/PyTorch-with-Examples-2024/blob/parts/train_en_azb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets transformers[sentencepiece] sacrebleu -q
from transformers import T5Tokenizer
from transformers import AutoTokenizer, T5ForConditionalGeneration

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# New way of making machine translation using Ali's idea

In [1]:
!pip install datasets transformers[sentencepiece] sacrebleu -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 20.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
def tokenize_function(examples):
    # Tokenize without padding to minimize memory usage
    tokenized = tokenizer(
        examples["text"],
        max_length=512,
        truncation=True
    )
    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": tokenized["input_ids"],  # For autoencoding tasks
    }

In [11]:
# Load the tokenizer from Hugging Face Hub
tokenizer = T5Tokenizer.from_pretrained("jalilkartal/AZB_EN_48m")
# Define the special tokens
special_tokens_dict = {
    "additional_special_tokens": [
        "Translate English to South Azerbaijani:",
        "Translate South Azerbaijani to English:"
    ]
}

# Add special tokens to the tokenizer
num_added_tokens = tokenizer.add_special_tokens(special_tokens_dict)
print(f"Added {num_added_tokens} special tokens.")

tokenizer_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/893k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/84.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Added 2 special tokens.


In [13]:
from datasets import load_dataset

dataset = load_dataset("jalilkartal/azb_en_198mb")
dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/1148860 [00:00<?, ? examples/s]

Map:   0%|          | 0/41601 [00:00<?, ? examples/s]

In [14]:
from transformers import T5ForConditionalGeneration, T5Config,DataCollatorForSeq2Seq

# Define the model configuration
model_config = T5Config(
    vocab_size=32102,
    n_positions=512,
    d_model=64,
    d_kv=64,
    d_ff=2048,
    num_layers=2,
    num_heads=4,
    relative_attention_num_buckets=16,
    dropout_rate=0.1,
    layer_norm_epsilon=1e-06,
    initializer_factor=1.0,
    is_encoder_decoder=True,
    pad_token_id=0,               # Padding token
    eos_token_id=1,               # End-of-sequence token
    decoder_start_token_id=0      # Decoder start token set to pad_token_id
)

# Initialize the model
model = T5ForConditionalGeneration(config=model_config)



In [15]:
# Define the Data Collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,  # Ensures proper padding
    return_tensors="pt"  # Returns PyTorch tensors
)

In [16]:
import os
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

In [17]:
print(f"Decoder Start Token ID: {model.config.decoder_start_token_id}")
tokenizer.pad_token_id = 0
tokenizer.eos_token_id = 1
model.config.decoder_start_token_id = model.config.pad_token_id


Decoder Start Token ID: 0


In [18]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1148860
    })
    validation: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 41601
    })
})

In [19]:
from transformers import Trainer, TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ali_najafi",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    save_steps=10_000,
    save_total_limit=2,
    fp16=True,
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],  # Tokenized dataset
    eval_dataset=dataset["validation"],  # Tokenized validation set
    data_collator=data_collator,  # Add the data collator here
    tokenizer=tokenizer,  # Optional: For dynamic padding during evaluation
)

# Start training
trainer.train()
#new_model

# 9. Save
# Save the model and tokenizer to Google Drive
model.save_pretrained('/content/drive/MyDrive/ali_najafi')
tokenizer.save_pretrained('/content/drive/MyDrive/ali_najafi')


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-19-4bc0863115f5>:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,25.382500,6.401244
2,19.411600,3.991726
3,16.919900,2.696006
4,14.036600,1.281135


('/content/drive/MyDrive/ali_najafi/tokenizer_config.json',
 '/content/drive/MyDrive/ali_najafi/special_tokens_map.json',
 '/content/drive/MyDrive/ali_najafi/spiece.model',
 '/content/drive/MyDrive/ali_najafi/added_tokens.json')

In [11]:
from datasets import load_dataset

dataset = load_dataset("Kartal-Ol/en-azb-548k")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

train.parquet:   0%|          | 0.00/87.9M [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/548900 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [21]:

tokenizer = T5Tokenizer.from_pretrained('/content/drive/MyDrive/ali_najafi')
new_model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/ali_najafi")


In [22]:
# Tokenize function for batched tokenization
def tokenize_function(examples):
    # Extract 'en' and 'azb' for each example in the batch
    source_texts = [example['en'] for example in examples['translation']]
    target_texts = [example['azb'] for example in examples['translation']]

    # Tokenize both source (English) and target (Azerbaijani Arabic script)
    source = tokenizer(source_texts, padding="max_length", truncation=True, max_length=128)
    target = tokenizer(target_texts, padding="max_length", truncation=True, max_length=128)

    return {
        'input_ids': source['input_ids'],
        'attention_mask': source['attention_mask'],
        'labels': target['input_ids']
    }

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/548900 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [23]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ali_najafifinetune",          # Where to save model checkpoints
    evaluation_strategy="epoch",    # Evaluate at the end of every epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=44, # Batch size per GPU
    per_device_eval_batch_size=44,  # Batch size for validation
    num_train_epochs=10,             # Number of epochs
    weight_decay=0.01,              # Weight decay for regularization
    save_total_limit=3,
    report_to="none" # Keep only the last 3 checkpoints
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [24]:

print("Special Tokens:", tokenizer.additional_special_tokens)
print("Token IDs:")
print(f"'Translate English to South Azerbaijani:' -> {tokenizer.convert_tokens_to_ids('Translate English to South Azerbaijani:')}")
print(f"'Translate South Azerbaijani to English:' -> {tokenizer.convert_tokens_to_ids('Translate South Azerbaijani to English:')}")


Special Tokens: ['Translate English to South Azerbaijani:', 'Translate South Azerbaijani to English:', '<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', '<extra_id_42>', '<extra_id_43>', '<extra_id_44>', '<extra_id_45>', '<extra_id_46>', '<extra_id_47>', '<extra_id_48>', '<extra_id_49>', '<extra_id_50>', '<extra_id_51>', '<extra_id_52>', '<extr

In [25]:
from transformers import Trainer
import os

# Create the trainer
trainer = Trainer(
    model=new_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

# Start training
trainer.train()

# 9. Save
# Save the model and tokenizer to Google Drive
model.save_pretrained('/content/drive/MyDrive/ali_najafifinetune')
tokenizer.save_pretrained('/content/drive/MyDrive/ali_najafifinetune')

Epoch,Training Loss,Validation Loss
1,1.330300,2.654588
2,1.259900,2.618233
3,1.227800,2.598120
4,1.211500,2.580761
5,1.190900,2.565206
6,1.169800,2.551982
7,1.181800,2.541846
8,1.166600,2.534170
9,1.151200,2.530668
10,1.162300,2.528943


('/content/drive/MyDrive/ali_najafifinetune/tokenizer_config.json',
 '/content/drive/MyDrive/ali_najafifinetune/special_tokens_map.json',
 '/content/drive/MyDrive/ali_najafifinetune/spiece.model',
 '/content/drive/MyDrive/ali_najafifinetune/added_tokens.json')

In [3]:
# Load fine-tuned model and tokenizer
model_path = "/content/drive/MyDrive/ali_najafifinetune"
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)


In [4]:
test_examples = [
    {"en": "How are you?", "azb": "سن نئجه‌سن؟"},
    {"en": "The weather is beautiful today.", "azb": "بوگون هوا گؤزل‌دیر."},
    {"en": "I am learning machine translation.", "azb": "من ماشین ترجمه اوْیره‌نیرم."}
]


In [9]:
def translate(text, source_prefix="Translate English to South Azerbaijani:"):
    # Add the source prefix for the model
    input_text = source_prefix + text

    # Tokenize the input text
    input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True)

    # Generate the output
    output_ids = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)

    # Decode the output to get the translated text
    translated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return translated_text


In [15]:
# Loop through test examples and print translations
for example in test_examples:
    source_text = example["en"]
    expected_translation = example["azb"]

    # Generate translation
    generated_translation = translate(source_text)

    print(f"Source (EN): {source_text}")
    print(f"Expected (AZB): {expected_translation}")
    print(f"Generated (AZB): {generated_translation}")
    print("-" * 50)


Source (EN): How are you?
Expected (AZB): سن نئجه‌سن؟
Generated (AZB): How.? are you are How
--------------------------------------------------
Source (EN): The weather is beautiful today.
Expected (AZB): بوگون هوا گؤزل‌دیر.
Generated (AZB): The Mechanism today The بحث is weather today. بیز satisfactory is weather today Theدَییشیب.. بیز satisfactory is weather today Theدَییشیب.. . The
--------------------------------------------------
Source (EN): I am learning machine translation.
Expected (AZB): من ماشین ترجمه اوْیره‌نیرم.
Generated (AZB): I am learning. learning am learning am learning machine translation machine.
--------------------------------------------------


In [27]:
dataset['train']['translation'][400000]['en']

'Many refuse to go out at night or to let their children play outside unattended \u200b — day or night .'

In [28]:
dataset['train']['translation'][400000]['azb']

'چوخلاری آخشاملار ائودن باییرا چیٛخماغا قوْرخور . والیدئینلر ، گۆنون هانسی ساعاتیندان آسیلی اوْلمایاراق ، اوُشاقلارینی حیطده اوْینایارکن نظارتسیز قوْیمورلار .'